In [2]:
import pandas as pd

# Load dataset
nav = pd.read_csv("data/raw/02_nav_history.csv")

print("Original Shape:", nav.shape)

# 1. Parse dates to datetime
nav["date"] = pd.to_datetime(nav["date"], errors="coerce")

# Check invalid dates
print("\nInvalid Dates:")
print(nav["date"].isnull().sum())

# 2. Sort by amfi_code and date
nav = nav.sort_values(by=["amfi_code", "date"])

# 3. Forward-fill missing NAV values
nav["nav"] = nav.groupby("amfi_code")["nav"].ffill()

# 4. Remove duplicates
duplicates = nav.duplicated().sum()
print("\nDuplicate Rows:", duplicates)

nav = nav.drop_duplicates()

# 5. Validate NAV > 0
invalid_nav = nav[nav["nav"] <= 0]

print("\nInvalid NAV Records:")
print(invalid_nav)

# Keep only valid NAV values
nav = nav[nav["nav"] > 0]

print("\nCleaned Shape:", nav.shape)

# Save cleaned dataset
nav.to_csv(
    "data/processed/nav_history_clean.csv",
    index=False
)

print("\nCleaned file saved successfully!")

Original Shape: (46000, 3)

Invalid Dates:
0

Duplicate Rows: 0

Invalid NAV Records:
Empty DataFrame
Columns: [amfi_code, date, nav]
Index: []

Cleaned Shape: (46000, 3)

Cleaned file saved successfully!


Clean investor_transactions.csv — standardise transaction_type values (SIP/Lumpsum/Redemption), validate amount > 0, fix date formats, check KYC status enum values.

In [4]:

# Clean investor_transactions.csv

import pandas as pd

# Load dataset
txn = pd.read_csv("data/raw/08_investor_transactions.csv")

print("Original Shape:", txn.shape)

# 1. Standardize transaction_type values
txn["transaction_type"] = (
    txn["transaction_type"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Map different values to standard values
txn["transaction_type"] = txn["transaction_type"].replace({
    "SYSTEMATIC INVESTMENT PLAN": "SIP",
    "SIP": "SIP",
    "LUMP SUM": "LUMPSUM",
    "LUMPSUM": "LUMPSUM",
    "PURCHASE": "LUMPSUM",
    "REDEMPTION": "REDEMPTION",
    "REDEEM": "REDEMPTION"
})

print("\nUnique Transaction Types:")
print(txn["transaction_type"].unique())

# 2. Validate amount > 0
invalid_amount = txn[txn["amount_inr"] <= 0]

print("\nInvalid Amount Records:")
print(invalid_amount.shape[0])

# Keep only valid transactions
txn = txn[txn["amount_inr"] > 0]

# 3. Fix date formats
txn["transaction_date"] = pd.to_datetime(
    txn["transaction_date"],
    errors="coerce"
)

print("\nInvalid Dates:")
print(txn["transaction_date"].isnull().sum())

# Remove rows with invalid dates
txn = txn.dropna(subset=["transaction_date"])

# 4. Check KYC status enum values
txn["kyc_status"] = (
    txn["kyc_status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

valid_kyc = ["VERIFIED", "PENDING", "REJECTED"]

invalid_kyc = txn[
    ~txn["kyc_status"].isin(valid_kyc)
]

print("\nInvalid KYC Records:")
print(invalid_kyc.shape[0])

if not invalid_kyc.empty:
    print(invalid_kyc[["kyc_status"]].drop_duplicates())

print("\nCleaned Shape:", txn.shape)

# Save cleaned dataset
txn.to_csv(
    "data/processed/investor_transactions_clean.csv",
    index=False
)

print("\nCleaned investor_transactions saved successfully!")

Original Shape: (32778, 13)

Unique Transaction Types:
<StringArray>
['SIP', 'REDEMPTION', 'LUMPSUM']
Length: 3, dtype: str

Invalid Amount Records:
0

Invalid Dates:
0

Invalid KYC Records:
0

Cleaned Shape: (32778, 13)

Cleaned investor_transactions saved successfully!


Clean scheme_performance.csv — validate all return values are numeric, flag anomalies, check expense_ratio range (0.1% – 2.5%).

In [7]:

# Clean scheme_performance.csv

import pandas as pd

# Load dataset
perf = pd.read_csv("data/raw/07_scheme_performance.csv")

print("Original Shape:", perf.shape)

# Return columns 
return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct"
]

# Convert returns to numeric
for col in return_cols:
    perf[col] = pd.to_numeric(
        perf[col],
        errors="coerce"
    )

# Find anomalies (non-numeric values)
anomalies = perf[
    perf[return_cols].isnull().any(axis=1)
]

print("\nAnomalies Found:")
print(anomalies)

# Expense ratio validation
invalid_expense = perf[
    (perf["expense_ratio_pct"] < 0.1) |
    (perf["expense_ratio_pct"] > 2.5)
]

print("\nInvalid Expense Ratio Records:")
print(invalid_expense)

# Save cleaned dataset
perf.to_csv(
    "data/processed/scheme_performance_clean.csv",
    index=False
)

print("\nCleaned file saved successfully!")

Original Shape: (40, 19)

Anomalies Found:
Empty DataFrame
Columns: [amfi_code, scheme_name, fund_house, category, plan, return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade]
Index: []

Invalid Expense Ratio Records:
Empty DataFrame
Columns: [amfi_code, scheme_name, fund_house, category, plan, return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade]
Index: []

Cleaned file saved successfully!


In [8]:
import pandas as pd
from glob import glob
import os

# Raw CSV files
csv_files = glob("data/raw/*.csv")

print("Total files found:", len(csv_files))

for file in csv_files:

    print("\nProcessing:", os.path.basename(file))

    # Load dataset
    df = pd.read_csv(file)

    # Remove duplicate rows
    df = df.drop_duplicates()

    # Remove completely empty rows
    df = df.dropna(how="all")

    # Standardize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    # Convert columns containing 'date' to datetime
    for col in df.columns:
        if "date" in col:
            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )

    # Remove rows where all values are missing
    df = df.dropna(how="all")

    # Save cleaned file
    filename = os.path.basename(file)

    output_file = (
        "data/processed/" +
        filename.replace(".csv", "_clean.csv")
    )

    df.to_csv(output_file, index=False)

    print("Saved:", output_file)
    print("Shape:", df.shape)

print("\nAll datasets cleaned successfully!")

Total files found: 16

Processing: 01_fund_master.csv
Saved: data/processed/01_fund_master_clean.csv
Shape: (40, 15)

Processing: 02_nav_history.csv
Saved: data/processed/02_nav_history_clean.csv
Shape: (46000, 3)

Processing: 03_aum_by_fund_house.csv
Saved: data/processed/03_aum_by_fund_house_clean.csv
Shape: (90, 5)

Processing: 04_monthly_sip_inflows.csv
Saved: data/processed/04_monthly_sip_inflows_clean.csv
Shape: (48, 6)

Processing: 05_category_inflows.csv
Saved: data/processed/05_category_inflows_clean.csv
Shape: (144, 3)

Processing: 06_industry_folio_count.csv
Saved: data/processed/06_industry_folio_count_clean.csv
Shape: (21, 6)

Processing: 07_scheme_performance.csv
Saved: data/processed/07_scheme_performance_clean.csv
Shape: (40, 19)

Processing: 08_investor_transactions.csv
Saved: data/processed/08_investor_transactions_clean.csv
Shape: (32778, 13)

Processing: 09_portfolio_holdings.csv
Saved: data/processed/09_portfolio_holdings_clean.csv
Shape: (322, 8)

Processing: 10_b

C:\Users\HP\AppData\Local\Temp\ipykernel_19100\2165193927.py:34: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(
C:\Users\HP\AppData\Local\Temp\ipykernel_19100\2165193927.py:34: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(
C:\Users\HP\AppData\Local\Temp\ipykernel_19100\2165193927.py:34: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(
C:\Users\HP\AppData\Local\Temp\ipykernel_19100\2165193927.py:34: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = 